# Previsão de fluxo financeiro, sazonalidade e detecção de anomalias

In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

FAST=os.environ.get("FINANCE_BI_FAST_TEST")=="1"
CSV=Path(os.environ.get("FINANCE_BI_BANKING_CSV","data/raw/indian_banking_transactions.csv"))
OUT=Path(os.environ.get("FINANCE_BI_OUTPUT_DIR","outputs"))/"04_fluxo_anomalias"
OUT.mkdir(parents=True,exist_ok=True)
assert CSV.is_file(),f"CSV não encontrado: {CSV}"
colunas=["transaction_id","transaction_date","transaction_direction","transaction_amount","transaction_status"]
df=pd.read_csv(CSV,usecols=colunas,parse_dates=["transaction_date"],low_memory=False).drop_duplicates("transaction_id")
df["transaction_amount"]=pd.to_numeric(df["transaction_amount"],errors="coerce")
df=df.dropna(subset=["transaction_date","transaction_amount"])
df=df.loc[df["transaction_status"].eq("Success")&df["transaction_amount"].ge(0)&df["transaction_direction"].isin(["Credit","Debit"])].copy()
assert not df.empty
meses=pd.date_range(df["transaction_date"].min().to_period("M").to_timestamp(),df["transaction_date"].max().to_period("M").to_timestamp(),freq="MS")
assert len(meses)>=36,"Requer ao menos 36 meses para previsão sazonal."
df["mes"]=df["transaction_date"].dt.to_period("M").dt.to_timestamp()
mov=df.pivot_table(index="mes",columns="transaction_direction",values="transaction_amount",aggfunc="sum",fill_value=0).reindex(meses,fill_value=0)
for c in ["Credit","Debit"]:
    if c not in mov.columns:mov[c]=0.
mov=mov[["Credit","Debit"]].astype(float)
mov["saldo_fluxo_inr"]=mov["Credit"]-mov["Debit"]
mov["movimentacao_bruta_inr"]=mov["Credit"]+mov["Debit"]
mov["transacoes"] = df.groupby("mes").size().reindex(meses,fill_value=0)
mov["ticket_medio_inr"]=mov["movimentacao_bruta_inr"]/mov["transacoes"].replace(0,np.nan)
mov["mes_do_ano"]=mov.index.month
for c in ["Credit","Debit","movimentacao_bruta_inr"]:
    mov[c+"_mom"]=mov[c].pct_change(fill_method=None).replace([np.inf,-np.inf],np.nan)
    mov[c+"_yoy"]=mov[c].pct_change(12,fill_method=None).replace([np.inf,-np.inf],np.nan)
print("Fluxos em INR: positivos e negativos NÃO são saldos bancários; representam movimentações concluídas.")
display(mov.tail(12))

In [ ]:
fig,ax=plt.subplots(2,1,figsize=(13,8),sharex=True)
mov[["Credit","Debit"]].plot(ax=ax[0],title="Entradas e saídas mensais (INR)")
mov["saldo_fluxo_inr"].plot(ax=ax[1],title="Fluxo líquido mensal (entradas - saídas)")
for a in ax:a.axhline(0,color="gray",linewidth=.7);a.grid(alpha=.25)
plt.tight_layout();plt.show()
sazonal=mov.groupby("mes_do_ano")[["Credit","Debit"]].mean()
sazonal.plot(kind="bar",figsize=(12,4),title="Sazonalidade mensal média (INR)")
plt.tight_layout();plt.show()

In [ ]:
def xgb_forecast(y,h):
    a=np.asarray(y,dtype=float)
    X=[];target=[]
    for t in range(12,len(a)):
        date=meses[0]+pd.DateOffset(months=t)
        X.append([a[t-1],a[t-2],a[t-3],a[t-6],a[t-12],np.mean(a[t-3:t]),np.sin(2*np.pi*date.month/12),np.cos(2*np.pi*date.month/12)])
        target.append(a[t])
    model=XGBRegressor(n_estimators=55 if FAST else 140,max_depth=3,learning_rate=.06,random_state=42,n_jobs=2,objective="reg:squarederror")
    model.fit(np.array(X,float),np.array(target,float))
    hist=list(a);results=[]
    for _ in range(h):
        date=meses[0]+pd.DateOffset(months=len(hist))
        x=[[hist[-1],hist[-2],hist[-3],hist[-6],hist[-12],np.mean(hist[-3:]),np.sin(2*np.pi*date.month/12),np.cos(2*np.pi*date.month/12)]]
        z=max(0.,float(model.predict(np.asarray(x,float))[0]));hist.append(z);results.append(z)
    return np.asarray(results)

def sarima_forecast(y,h):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fit=SARIMAX(np.asarray(y,float),order=(1,1,1),seasonal_order=(0,1,1,12),enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=35 if FAST else 90)
    return np.maximum(0,np.asarray(fit.forecast(h),float))

def prever(nome,y,h):
    if nome=="SARIMA":return sarima_forecast(y,h)
    if nome=="XGBoost":return xgb_forecast(y,h)
    if nome=="Naive sazonal":return np.array(y[-12:][-h:],float)
    raise ValueError(nome)

origens=[len(meses)-3,len(meses)-6,len(meses)-9]
if FAST:origens=origens[:1]
linhas=[];falhas=[]
for direcao in ["Credit","Debit"]:
    a=mov[direcao].to_numpy(dtype=float)
    for corte in sorted(origens):
        treino,real=a[:corte],a[corte:corte+3]
        for modelo in ["Naive sazonal","SARIMA","XGBoost"]:
            try:
                pre=prever(modelo,treino,3)
                assert np.isfinite(pre).all()
                linhas.extend([dict(direcao=direcao,modelo=modelo,corte=str(meses[corte-1].date()),horizonte=k,real_inr=float(v),previsao_inr=float(p),erro_abs=abs(v-p)) for k,(v,p) in enumerate(zip(real,pre),1)])
            except Exception as exc:falhas.append((direcao,modelo,str(exc)[:180]))
assert linhas,"Nenhuma validação temporal foi concluída."
backtest=pd.DataFrame(linhas)
score=backtest.groupby(["direcao","modelo"],as_index=False).agg(mae=("erro_abs","mean"),soma_erro=("erro_abs","sum"),soma_real=("real_inr","sum"))
score["wape"]=np.where(score["soma_real"]>0,score["soma_erro"]/score["soma_real"],np.nan)
score=score.sort_values(["direcao","mae"])
display(score)
if falhas:print("Falhas de convergência/modelagem:",falhas)

In [ ]:
futuro=pd.date_range(meses[-1]+pd.offsets.MonthBegin(1),periods=3,freq="MS")
proj=pd.DataFrame({"mes":futuro})
for direcao in ["Credit","Debit"]:
    melhor=score.loc[score["direcao"].eq(direcao)].iloc[0]["modelo"]
    proj[direcao+"_previsto_inr"]=prever(melhor,mov[direcao].to_numpy(),3)
    print(direcao,"->",melhor)
proj["fluxo_liquido_previsto_inr"]=proj["Credit_previsto_inr"]-proj["Debit_previsto_inr"]
display(proj)
fig,axes=plt.subplots(1,2,figsize=(14,4))
for ax,direcao in zip(axes,["Credit","Debit"]):
    ax.plot(mov.index,mov[direcao],label="Histórico")
    ax.plot(proj["mes"],proj[direcao+"_previsto_inr"],"o--",label="Previsão")
    ax.set(title=direcao,ylabel="INR");ax.legend();ax.grid(alpha=.2)
plt.tight_layout();plt.show()

In [ ]:
# Anomalias são mensais e estatísticas: NÃO equivalem a fraude.
base=mov[["Credit","Debit","saldo_fluxo_inr","movimentacao_bruta_inr","transacoes","ticket_medio_inr"]].copy()
base["ticket_medio_inr"]=base["ticket_medio_inr"].fillna(0)
base["variacao_bruta_abs"]=mov["movimentacao_bruta_inr"].diff().abs().fillna(0)
idx_corte=max(24,int(len(base)*.75))
base_train=base.iloc[:idx_corte]
scaler=RobustScaler().fit(base_train)
Z_train=scaler.transform(base_train)
Z_all=scaler.transform(base)
modelo_anomalia=IsolationForest(n_estimators=80 if FAST else 240,contamination=.10,random_state=42,n_jobs=2).fit(Z_train)
score_anomalia=modelo_anomalia.decision_function(Z_all)
resultado=mov[["Credit","Debit","saldo_fluxo_inr","movimentacao_bruta_inr"]].copy()
resultado["score_anomalia"]=score_anomalia
resultado["anomalia"]=modelo_anomalia.predict(Z_all)==-1
resultado["periodo"] = np.where(np.arange(len(base))<idx_corte,"treino","avaliacao_futura")
print("Meses sinalizados como atípicos em janela posterior à calibração:")
display(resultado.loc[resultado["anomalia"] & resultado["periodo"].eq("avaliacao_futura")])
fig,ax=plt.subplots(figsize=(13,4))
ax.plot(resultado.index,resultado["movimentacao_bruta_inr"],label="Volume bruto")
mask=resultado["anomalia"]
ax.scatter(resultado.index[mask],resultado.loc[mask,"movimentacao_bruta_inr"],color="tab:red",label="Anomalia estatística",zorder=3)
ax.axvline(base.index[idx_corte],linestyle="--",color="gray",label="Início da avaliação")
ax.set(title="Detecção de anomalias mensais com Isolation Forest",ylabel="INR");ax.grid(alpha=.2);ax.legend();plt.tight_layout();plt.show()

In [ ]:
mov.to_csv(OUT/"fluxo_mensal.csv",index_label="mes")
backtest.to_csv(OUT/"validacao_temporal_fluxo.csv",index=False)
score.to_csv(OUT/"metricas_previsao_fluxo.csv",index=False)
proj.to_csv(OUT/"previsao_proximos_3_meses.csv",index=False)
resultado.to_csv(OUT/"anomalias_mensais.csv",index_label="mes")
print("Arquivos:",*[str(p) for p in sorted(OUT.glob('*.csv'))],sep="\n")
print("Dados sintéticos e sem inferência causal. Projeções de entradas e saídas não representam liquidez disponível de uma instituição.")